# Notebook 09 · Versioned pilot adjudication and rescoring

CPU only. Attach the exact published Stage 03–08 pilot releases already used in the completed pilot, plus the supplied `evaluator_unavailable_review_completed.csv`. Do **not** regenerate Stage 07 and do **not** edit generated source.

This versioned correction applies the pipeline's existing generated-output failure rules to the 15 reviewed generated-test defects. It preserves the single genuine evaluator limitation as unavailable. The correction is applied uniformly during scoring, retains the original execution evidence, emits a complete audit, and never converts a generated failure into a pass.

Run all cells. Publish the three files written to `/kaggle/working/stage_09_pilot_dataset` as a new Stage 09 Kaggle dataset. Do not mix it with the older Stage 09 release.


In [1]:
# Notebook 09 · Cell 01 · Verify releases and load registered runtime

from collections import Counter, defaultdict
from fractions import Fraction
import csv
import hashlib
import io
import json
import math
import os
from pathlib import Path, PurePosixPath
import platform
import random
import re
from statistics import NormalDist, mean, stdev
import types
import zipfile

import numpy as np
import scipy
from scipy.stats import norm, t
import yaml

INPUT = Path(os.environ.get("OPENAPI_INPUT", "/kaggle/input"))
WORK = Path(os.environ.get("OPENAPI_WORK", "/kaggle/working"))
STRATA = ("low", "medium", "high")
PILOT_VERSION = "stage08-fresh-pilot-v3-full-claim-review"
PILOT_HASHES = {
    "03": "fe26961c4fa5def49f7a3db13f2f9cd7f80e25f6717ab86a1e70f535b5d385fa",
    "04": "86ed5e55413d29d0a3d45b8270976424ddbbe74d48a0cfa9bca4be3992ba37d9",
    "05": "14698c5c34d1b693d062f993bb71d48b9f9b73d57d16c07613519dbda111ad0d",
    "06": "d47e519366bfe44ea9364f9539da838d221aff686215ddeb141eeb84f462b902",
    "07": "3da3c1a7cca4cdd344d6cbaa5f1564632dc453cb937fcd8e38ff935c1836b1bb",
    "08": "c3e71cfcb6d5ba071eab95fe159fb3a4a31a8184e4d3f38341292e45c3bb8136",
}
MAIN_PRODUCERS = {"09": "09_pilot_scoring", "10": "10_power_and_main_sampling",
                  "11": "11_main_inference_and_report"}


def require(condition, message):
    if not condition:
        raise RuntimeError(message)


def sha(data):
    return hashlib.sha256(data).hexdigest()


def file_hash(path):
    with path.open("rb") as stream:
        return hashlib.file_digest(stream, "sha256").hexdigest()


def json_bytes(data):
    return (json.dumps(data, sort_keys=True, ensure_ascii=False, allow_nan=False,
                       separators=(",", ":")) + "\n").encode("utf-8")


def digest(data):
    return sha(json_bytes(data).rstrip(b"\n"))


def csv_rows(data):
    return list(csv.DictReader(io.StringIO(data.decode("utf-8-sig"))))


def jsonl_rows(data):
    return [json.loads(line) for line in data.splitlines() if line.strip()]


def csv_bytes(rows, fields=None):
    buffer = io.StringIO(newline="")
    writer = csv.DictWriter(buffer, fieldnames=fields or list(rows[0]), lineterminator="\n")
    writer.writeheader()
    writer.writerows(rows)
    return buffer.getvalue().encode("utf-8")


def keyed(rows, key, description):
    mapping = {key(row): row for row in rows}
    require(len(mapping) == len(rows), f"Duplicate identity in {description}")
    return mapping


def group(rows, key):
    result = defaultdict(list)
    for row in rows:
        result[key(row)].append(row)
    return result


def path_name(value):
    path = PurePosixPath(value)
    require(value not in ("", ".") and not path.is_absolute() and ".." not in path.parts
            and "\\" not in value and str(path) == value, f"Invalid artifact path {value!r}")
    return value


class Release:
    def __init__(self, stage, partition=None, pin=None):
        suffix = "_main" if partition == "main" and stage in {"05", "06", "07", "08"} else ""
        self.stage = stage
        self.archive = None
        names = sorted(INPUT.rglob(f"stage_{stage}{suffix}_manifest.json"))
        matches = []
        for path in names:
            data = path.read_bytes()
            meta = json.loads(data)
            if pin and sha(data) != pin:
                continue
            if partition and meta.get("partition") != partition:
                continue
            matches.append((path, data, meta))
        require(len(matches) == 1, f"Stage {stage} {partition or ''}: expected one matching manifest, found {len(matches)}")
        path, data, self.external = matches[0]
        require(self.external.get("stage") == stage and self.external.get("status") == "PASS",
                f"Stage {stage} input has not passed")
        self.external_hash = sha(data)
        description = self.external.get("archive")
        name = description["filename"] if isinstance(description, dict) else description
        name = name or self.external["archive_name"]
        require(Path(name).name == name, "Invalid archive name")
        archive = path.parent / name
        relative = f"registries/{stage}_manifest.json"
        self.internal_hash = self.external["internal_manifest_sha256"]
        if archive.is_file():
            require(file_hash(archive) == self.external["archive_sha256"], f"Stage {stage} archive hash differs")
            self.archive = zipfile.ZipFile(archive)
            names = self.archive.namelist()
            candidates = [n for n in names if n == relative or n.endswith("/" + relative)]
            require(len(set(names)) == len(names) and len(candidates) == 1, f"Stage {stage} archive layout differs")
            self.prefix = candidates[0][:-len(relative)]
            manifest_bytes = self.archive.read(candidates[0])
        else:
            candidates = [p for p in path.parent.rglob(relative)
                          if p.is_file() and file_hash(p) == self.internal_hash]
            require(len(candidates) == 1, f"Stage {stage} missing or ambiguous expanded manifest")
            self.root = candidates[0].parent.parent
            manifest_bytes = candidates[0].read_bytes()
        require(sha(manifest_bytes) == self.internal_hash, f"Stage {stage} internal manifest differs")
        self.internal = json.loads(manifest_bytes)
        require(self.internal["stage"] == stage and self.internal["status"] == "PASS", f"Stage {stage} internal status differs")
        if partition:
            require(self.internal["partition"] == partition, f"Stage {stage} partition differs")
        def artifact_name(row):
            identities = {row[key] for key in ("artifact_path", "relative_path", "path") if key in row}
            require(len(identities) == 1, f"Stage {stage} inconsistent artifact path")
            return path_name(identities.pop())
        self.artifacts = keyed(self.internal["artifacts"], artifact_name, f"Stage {stage} manifest")
        require(len(self.artifacts) == self.internal["artifact_count"], f"Stage {stage} artifact count differs")
        for n, record in self.artifacts.items():
            require(record["artifact_path"] == n and record["schema_version"] == "openapi-artifact-v1"
                    and bool(record["producer_notebook"])
                    and record["config_hash"] == self.internal["configuration_hash"]
                    and record["upstream_manifest_hash"] == digest(self.internal["upstream_manifest_sha256"]),
                    f"Stage {stage} artifact provenance differs: {n}")
        self.checked = set()

    def read(self, relative):
        record = self.artifacts[relative]
        data = self.archive.read(self.prefix + relative) if self.archive else (self.root / relative).read_bytes()
        require(len(data) == record["bytes"] and sha(data) == record["sha256"],
                f"Stage {self.stage}: artifact differs at {relative}")
        self.checked.add(relative)
        return data

    def csv(self, relative):
        return csv_rows(self.read(relative))

    def jsonl(self, relative):
        return jsonl_rows(self.read(relative))

    def json(self, relative):
        return json.loads(self.read(relative))

    def yaml(self, relative):
        return yaml.safe_load(self.read(relative))

    def link(self, other):
        require(self.internal["upstream_manifest_sha256"].get(other.stage) == other.internal_hash,
                f"Stage {self.stage} does not match the attached Stage {other.stage}")


def published_pilot(stage):
    # An amended Stage 08 must be explicitly pinned by its external manifest hash.
    pin = os.environ.get("OPENAPI_PILOT08_SHA256", PILOT_HASHES[stage]) if stage == "08" else PILOT_HASHES[stage]
    return Release(stage, "pilot" if stage in {"06", "07", "08"} else None, pin)


def verify_links(releases):
    for release in releases.values():
        for stage in release.internal["upstream_manifest_sha256"]:
            if stage in releases:
                release.link(releases[stage])


def load_module(name, data):
    # Load only registered research modules; the analysis never runs generated tests.
    module = types.ModuleType(name)
    exec(compile(data, f"<registered {name}>", "exec", dont_inherit=True), module.__dict__)
    return module


def runtime():
    return {"python": platform.python_version(), "numpy": np.__version__,
            "scipy": scipy.__version__, "pyyaml": yaml.__version__}


def write_release(stage, partition, files, sources, metadata):
    destination = WORK / f"stage_{stage}_{partition}_dataset"
    destination.mkdir(parents=True, exist_ok=True)
    config_hash = digest({n: sha(b) for n, b in files.items() if n.startswith(("config/", "src/"))})
    upstream = {s: release.internal_hash for s, release in sources.items()}
    provenance = {"schema_version": "openapi-artifact-v1", "producer_notebook": MAIN_PRODUCERS[stage],
                  "config_hash": config_hash, "upstream_manifest_hash": digest(upstream)}
    artifacts = [{"artifact_path": n, "relative_path": n, "bytes": len(files[n]),
                  "sha256": sha(files[n]), **provenance} for n in sorted(files)]
    base = {"stage": stage, "status": "PASS", "partition": partition,
            "design_id": sources["06"].internal["design_id"], **metadata}
    internal = json_bytes({**base, "artifacts": artifacts, "artifact_count": len(artifacts),
                           "artifact_record_schema": "openapi-artifact-v1",
                           "configuration_hash": config_hash, "upstream_manifest_sha256": upstream})
    filename = f"stage_{stage}_{partition}.zip"
    prefix = f"stage_{stage}_{partition}/prose_operation_experiment"
    internal_path = f"registries/{stage}_manifest.json"
    tmp = destination / (filename + ".tmp")
    with zipfile.ZipFile(tmp, "w", zipfile.ZIP_DEFLATED, compresslevel=7) as archive:
        for n, b in sorted({**files, internal_path: internal}.items()):
            entry = zipfile.ZipInfo(prefix + "/" + n, (2026, 11, 11, 0, 0, 0))
            archive.writestr(entry, b, zipfile.ZIP_DEFLATED, 7)
    with zipfile.ZipFile(tmp) as archive:
        require(archive.testzip() is None and len(archive.namelist()) == len(artifacts) + 1
                and archive.read(prefix + "/" + internal_path) == internal
                and all(sha(archive.read(prefix + "/" + a["artifact_path"])) == a["sha256"] for a in artifacts),
                "Release archive verification failed")
    archive_path = destination / filename
    archive_hash = file_hash(tmp)
    manifest = json_bytes({**base, "archive": {"filename": filename, "project_prefix": prefix,
                                               "sha256": archive_hash, "bytes": tmp.stat().st_size,
                                               "member_count": len(artifacts) + 1},
                           "archive_sha256": archive_hash, "internal_manifest_sha256": sha(internal),
                           "internal_manifest": {"relative_path": internal_path,
                                                 "artifact_count": len(artifacts), "sha256": sha(internal)}})
    manifest_name = f"stage_{stage}_manifest.json"
    proposed = {manifest_name: manifest, f"stage_{stage}_checksums.sha256": (
        f"{archive_hash}  {filename}\n{sha(manifest)}  {manifest_name}\n").encode()}
    try:
        require(not archive_path.exists() or file_hash(archive_path) == archive_hash,
                f"Existing release differs at {archive_path}. Use a new OPENAPI_WORK directory.")
        for n, b in proposed.items():
            require(not (destination / n).exists() or (destination / n).read_bytes() == b,
                    f"Existing release differs at {destination / n}")
        tmp.replace(archive_path)
        for n, b in proposed.items():
            (destination / n).write_bytes(b)
    finally:
        tmp.unlink(missing_ok=True)
    print(f"Stage {stage} {partition} release verified at {destination}")
    return destination


def require_math(actual, expected, message):
    require(math.isclose(float(actual), float(expected), rel_tol=1e-9, abs_tol=1e-12), message)


In [2]:
# Notebook 09 · Cell 02 · Reconcile and score the complete pilot ledger

EVIDENCE = "data/evidence/"


def load_unavailable_review():
    matches = sorted(INPUT.rglob("evaluator_unavailable_review_completed.csv"))
    require(len(matches) == 1,
            "Attach exactly one evaluator_unavailable_review_completed.csv")
    raw = matches[0].read_bytes()
    rows = csv_rows(raw)
    required = {
        "task_id", "op_id", "artifact_id", "collection_status", "baseline_status",
        "request_ids", "reviewed_scenario_id", "baseline_id",
        "spec_evidence_pointer", "oracle_evidence_pointer", "decision",
        "rationale", "recommended_action",
    }
    require(rows and required <= set(rows[0]), "Unavailable review schema differs")
    decisions = Counter(row["decision"] for row in rows)
    require(len(rows) == 16 and decisions == Counter(
        TEST_CONSTRUCTION_DEFECT=15, EVALUATOR_LIMITATION=1),
        "Expected the completed 15 generated-test / 1 evaluator-limitation adjudication")
    require(all(row["task_id"] and row["op_id"] and row["artifact_id"]
                and row["request_ids"] and row["spec_evidence_pointer"]
                and row["oracle_evidence_pointer"] and row["rationale"]
                and row["recommended_action"]
                and row["reviewed_scenario_id"] and row["baseline_id"]
                for row in rows), "Unavailable review contains blank required evidence")
    return keyed(rows, lambda row: row["task_id"], "unavailable adjudication"), raw


def reconcile_review(evaluation, claims):
    require(evaluation.internal.get("checkpoint_c_complete") is True
            and evaluation.external.get("checkpoint_c_complete") is True,
            "Stage 08 has not completed Checkpoint C")
    by_blind = keyed(claims, lambda r: r["blind_claim_id"], "reviewed claims")
    require(all(k and r["review_status"] == "RECONCILED" for k, r in by_blind.items()), "Incomplete assertion review")
    r1, r2, r3 = [keyed(evaluation.csv(f"reviews/completed/reviewer_{i}.csv"),
                        lambda r: r["blind_claim_id"], f"Reviewer {i}") for i in (1, 2, 3)]
    sample = evaluation.json(EVIDENCE + "reviewer_2_frozen_sample.json")
    expected = sorted(random.Random(sample["seed"]).sample(sorted(by_blind), min(len(claims), 300)))
    require(sample["sampled_ids"] == expected and r2.keys() == set(expected)
            and r1.keys() == by_blind.keys(), "Incorrect frozen review coverage")
    disagreements = {k for k in r2 if r1[k]["label"] != r2[k]["label"]}
    require(r3.keys() == disagreements and all(r["adjudication_status"] == "COMPLETED" for r in r3.values()),
            "Incomplete Reviewer 3 decisions")
    labels = {"supported", "contradicted", "undocumented_or_unsupported", "ambiguous_or_unverifiable"}
    for number, decisions in enumerate((r1, r2, r3), 1):
        require(all(row["label"] in labels and row["reviewer_id"] == f"reviewer_{number}"
                    and row["reasoning"].strip() for row in decisions.values()), "Invalid recorded review")
    for k, row in by_blind.items():
        chosen = r3.get(k, r1[k])
        require(row["review_label"] == chosen["label"] and row["requirement_id"] == chosen["requirement_id"]
                and row["review_evidence_pointer"] == chosen["evidence_pointer"],
                "Final claim does not match the reconciled reviewer decision")
    internal = keyed(evaluation.jsonl(EVIDENCE + "assertion_claims_internal.jsonl"),
                     lambda r: r["claim_id"], "frozen assertion extraction")
    require(internal.keys() == {r["claim_id"] for r in claims}
            and all(all(row[k] == v for k, v in internal[row["claim_id"]].items()) for row in claims),
            "Frozen assertion frame differs from reviewed frame")


def expected_oracles(release, selection, partition):
    result = {}
    record_types = {"scenarios": "scenario_id", "baselines": "baseline_id",
                    "requirements": "requirement_id", "response_faults": "fault_id",
                    "legitimate_variations": "variation_id"}
    for op in selection:
        root = f"oracles/frozen/{partition}/{op}"
        records = {kind: release.jsonl(f"{root}/{kind}.jsonl") for kind in record_types}
        for kind, key in record_types.items():
            keyed(records[kind], lambda r: r[key], kind)
        records["faults"] = records["response_faults"]
        result[op] = records
    return result


def qualified_kill(row, fault, baseline, requests):
    if row["status"] != "KILLED_ASSERTION":
        return False
    target = row["replay_target"]
    attempt = requests.get(row.get("target_baseline_request_id"))
    item = target["pytest_item_id"]
    require(baseline["baseline_pass"] is True and row["killed"] is True and row["consumed"] is True
            and row["failure_attribution"] == "assertion" and attempt is not None
            and attempt["status"] == "DELIVERED" and attempt["task_id"] == row["task_id"]
            and attempt["pytest_item_id"] == item and item in row["attributed_assertion_item_ids"]
            and baseline["item_outcomes"].get(item) == "passed"
            and attempt.get("baseline_id") == fault["baseline_id"]
            and any(x.get("replay_mutation_applied") and x.get("fault_id") == fault["fault_id"]
                    and x.get("pytest_item_id") == item for x in row["request_attempts"]),
            "A claimed kill lacks a qualified assertion-linked replay")
    return True


def effective_check(claim, oracle, attempts, fault_rows):
    if claim["review_label"] != "supported" or claim["reachability"] != "REACHED":
        return None
    req = claim["requirement_id"]
    requirement = next((r for r in oracle["requirements"] if r["requirement_id"] == req), None)
    require(requirement is not None and requirement["source_pointer"] == claim["review_evidence_pointer"],
            "Supported claim lacks a corresponding frozen requirement")
    reached = set(claim["reached_item_ids"])
    delivered = [r for r in attempts if r["status"] == "DELIVERED"
                 and r["pytest_item_id"] in reached
                 and r.get("scenario_id") in claim["observed_scenario_ids"]]
    if not delivered:
        return None
    linked_faults = {r["fault_id"] for r in oracle["faults"] if r["requirement_id"] == req
                     and r.get("applicability") == "APPLICABLE"}
    for fault_id in sorted(linked_faults):
        row = fault_rows[fault_id]
        if row["status"] != "KILLED_ASSERTION":
            continue
        target = row["replay_target"]
        if not any(r["request_id"] == row["target_baseline_request_id"] for r in delivered):
            continue
        # Stage 08 names the failed item. Locate the actual failing source line
        # before crediting this particular assertion.
        location = re.compile(r"^work/generated_test\.py:" + str(claim["line"])
                              + r": in " + re.escape(target["pytest_item_id"].split("::")[-1].split("[")[0]) + r"\s*$",
                              re.MULTILINE)
        if location.search(row["stdout"]) and any(
                line.strip() == claim["source_span"].strip()
                for line in row["stdout"].splitlines() if line.strip().startswith("assert ")):
            return {"requirement_id": req, "reached": True, "grounded": True,
                    "linkage_qualified": True, "evidence_type": "assertion_linked_frozen_fault"}
    if linked_faults:
        return None
    # The registered semantic review can substantiate non-fault-linked checks
    # if the assertion has a single unambiguous observed response context.
    if len({(r["scenario_id"], r.get("response_status")) for r in delivered}) == 1:
        return {"requirement_id": req, "reached": True, "grounded": True,
                "linkage_qualified": True, "evidence_type": "reconciled_semantic_review"}
    return None


def score_partition(sources, selection, partition, scoring_bytes, unavailable_review):
    oracle, tasks, generation, evaluation = (sources[s] for s in ("05", "06", "07", "08"))
    for release in (generation, evaluation):
        require(release.read("config/metrics.yaml") == tasks.read("config/metrics.yaml")
                and release.read("config/analysis_registered.yaml") == tasks.read("config/analysis_registered.yaml"),
                "Frozen scoring or analysis contract differs")
    require(scoring_bytes == tasks.read("src/metrics.py") == evaluation.read("src/metrics.py"),
            "Scoring implementation differs from frozen Stage 06")
    metric = load_module("frozen_metrics", scoring_bytes)
    task_rows = keyed(tasks.csv("data/tasks.csv"), lambda r: r["task_id"], "tasks")
    artifact_rows = keyed(generation.csv("data/artifacts.csv"), lambda r: r["task_id"], "raw-generation ledger")
    execution = keyed(evaluation.csv(EVIDENCE + "execution.csv"), lambda r: r["task_id"], "execution")
    baseline = keyed(evaluation.jsonl(EVIDENCE + "baseline_results.jsonl"), lambda r: r["task_id"], "baseline")
    collection = keyed(evaluation.jsonl(EVIDENCE + "collection_results.jsonl"), lambda r: r["task_id"], "collection")
    require(task_rows.keys() == artifact_rows.keys() == execution.keys() == baseline.keys() == collection.keys(),
            "Expected task identities differ across Stages 06–08")
    unavailable_tasks = {task_id for task_id in task_rows
                         if execution[task_id]["collection_status"] == "EVALUATOR_UNAVAILABLE"
                         or execution[task_id]["baseline_status"] == "EVALUATOR_UNAVAILABLE"}
    require(set(unavailable_review) == unavailable_tasks,
            "Completed adjudication must cover every and only evaluator-unavailable task")
    for task_id, row in unavailable_review.items():
        require(row["op_id"] == task_rows[task_id]["op_id"]
                and row["artifact_id"] == artifact_rows[task_id]["artifact_id"]
                and row["collection_status"] == execution[task_id]["collection_status"]
                and row["baseline_status"] == execution[task_id]["baseline_status"],
                "Adjudication identity or frozen status differs")
    require(len(task_rows) == 2 * len(selection), "Expected one generation per condition and operation")
    for op, operation_tasks in group(task_rows.values(), lambda r: r["op_id"]).items():
        require(op in selection and len(operation_tasks) == 2
                and {r["condition"] for r in operation_tasks} == {"L0", "L1"}
                and {r["pair_id"] for r in operation_tasks} == {selection[op]["pair_id"]}
                and len({r["seed"] for r in operation_tasks}) == 1, "Invalid task pair or paired seed")
    require({r["op_id"] for r in task_rows.values()} == selection.keys(), "Task and selected-operation frames differ")
    require(all(r["partition"] == partition and str(r["replicate"]) == "1" for r in task_rows.values()),
            "Task partition or replicate differs")
    inventory = keyed(evaluation.jsonl("data/inventory/input_artifact_inventory.jsonl"),
                      lambda r: r["task_id"], "execution input inventory")
    require(inventory.keys() == task_rows.keys(), "Execution inventory differs")
    for k, row in artifact_rows.items():
        require(inventory[k]["raw_sha256"] == (row["raw_hash"] or None)
                and inventory[k]["extracted_sha256"] == (row["extracted_hash"] or None)
                and inventory[k]["extraction_status"] == row["extraction_status"],
                "Evaluation consumed different generation evidence")
    claims = evaluation.jsonl(EVIDENCE + "assertion_claims.jsonl")
    reconcile_review(evaluation, claims)
    by_task_claim = group(claims, lambda r: r["task_id"])
    requests = evaluation.jsonl(EVIDENCE + "request_attempts.jsonl")
    requests_by_id = keyed(requests, lambda r: r["request_id"], "request attempts")
    by_task_requests = group(requests, lambda r: r["task_id"])
    faults = evaluation.jsonl(EVIDENCE + "mutation_results.jsonl")
    variations = evaluation.jsonl(EVIDENCE + "variation_results.jsonl")
    frozen = keyed(evaluation.jsonl(EVIDENCE + "opportunities_frozen.jsonl"),
                   lambda r: r["opportunity_id"], "frozen opportunities")
    outcomes = keyed(evaluation.jsonl(EVIDENCE + "opportunity_outcomes.jsonl"),
                     lambda r: r["opportunity_id"], "opportunity results")
    require(frozen.keys() == outcomes.keys() and all(all(outcomes[k][f] == v for f, v in row.items())
            for k, row in frozen.items()), "Frozen replay ledger differs from its reconciliation")
    for kind, rows, id_key in (("fault", faults, "fault_id"), ("variation", variations, "variation_id")):
        lookup = keyed(rows, lambda r: r["opportunity_id"], f"{kind} results")
        require(lookup.keys() == {k for k, row in frozen.items() if row["kind"] == kind}
                and all(row["task_id"] == frozen[k]["task_id"]
                        and row[id_key] == frozen[k]["oracle_record_id"]
                        and row["status"] == outcomes[k]["final_status"] for k, row in lookup.items()),
                f"{kind} reconciliation mismatch")
    by_task_fault = group(faults, lambda r: r["task_id"])
    by_task_opportunity = group(frozen.values(), lambda r: r["task_id"])
    require(set(by_task_claim) | set(by_task_requests) <= task_rows.keys()
            and by_task_opportunity.keys() == task_rows.keys(), "Evidence contains unknown or missing tasks")
    oracles = expected_oracles(oracle, selection, partition)
    long, links = [], []
    for task_id, task in sorted(task_rows.items()):
        op = task["op_id"]
        answer = oracles[op]
        expected = {("baseline", r["baseline_id"]) for r in answer["baselines"]}
        expected |= {("scenario", r["scenario_id"]) for r in answer["scenarios"] if r.get("applicability", "APPLICABLE") == "APPLICABLE"}
        expected |= {("fault", r["fault_id"]) for r in answer["faults"] if r.get("applicability", "APPLICABLE") == "APPLICABLE"}
        expected |= {("variation", r["variation_id"]) for r in answer["legitimate_variations"] if r.get("applicability", "APPLICABLE") == "APPLICABLE"}
        actual = [(r["kind"], r["oracle_record_id"]) for r in by_task_opportunity[task_id]]
        require(len(actual) == len(expected) and set(actual) == expected
                and all(r["op_id"] == op for r in by_task_opportunity[task_id]), "Incomplete oracle opportunity frame")
        b, e = baseline[task_id], execution[task_id]
        require(e["baseline_status"] == b["status"] and e["collection_status"] == collection[task_id]["status"],
                "Execution summaries disagree with detailed results")
        require(b["status"] != "EVALUATOR_FAILURE" and e["collection_status"] != "EVALUATOR_FAILURE",
                "Evaluator failure requires a documented rerun")
        require(int(e["collected_item_count"] or 0) == len(collection[task_id]["item_ids"]),
                "Collected item count differs")
        adjudication = unavailable_review.get(task_id)
        generated_output_failure = bool(
            adjudication and adjudication["decision"] == "TEST_CONSTRUCTION_DEFECT")
        effective_collection_status = (
            "GENERATED_OUTPUT_FAILURE"
            if generated_output_failure and e["collection_status"] == "EVALUATOR_UNAVAILABLE"
            else e["collection_status"]
        )
        effective_baseline_status = (
            "GENERATED_OUTPUT_FAILURE"
            if generated_output_failure and b["status"] == "EVALUATOR_UNAVAILABLE"
            else ("COMPLETE" if b["status"] == "PASS" else b["status"])
        )
        seen = by_task_requests[task_id]
        require(all(r["op_id"] == op for r in seen), "Request evidence points to another operation")
        applicable_scenarios = {r["scenario_id"] for r in answer["scenarios"] if r.get("applicability", "APPLICABLE") == "APPLICABLE"}
        baseline_requests = []
        for r in seen:
            cl = r.get("classification") or {}
            scenario = r.get("scenario_id")
            classifiable = cl.get("classifiable") is True or r.get("reason_code") == "REQUEST_OUTSIDE_TARGET_OPERATION"
            baseline_requests.append({"attempt_id": r["request_id"], "classifiable": classifiable,
                                      "matched_intent": classifiable and scenario in applicable_scenarios,
                                      "scenario_id": scenario if classifiable else None})
        reviewed = by_task_claim[task_id]
        require(all(r["op_id"] == op and r["condition"] == task["condition"]
                    and r["artifact_id"] == artifact_rows[task_id]["artifact_id"] for r in reviewed),
                "Reviewed assertion points to another generation")
        listed_faults = {r["fault_id"]: r for r in answer["faults"] if r.get("applicability", "APPLICABLE") == "APPLICABLE"}
        results = keyed(by_task_fault[task_id], lambda r: r["fault_id"], "task fault results")
        effective = [link for claim in reviewed if (link := effective_check(claim, answer, seen, results))]
        links.extend({"task_id": task_id, **link} for link in effective)
        unavailable = {}
        if (not generated_output_failure
                and any(r["status"] == "EVALUATOR_UNAVAILABLE" for r in results.values())):
            unavailable["unconditional_response_fault_detection"] = "FAULT_OR_BASELINE_EVALUATOR_UNAVAILABLE"
        if (not generated_output_failure
                and any(r["reachability"] == "UNVERIFIED" for r in reviewed)
                and e["collection_status"] == "PASS"):
            unavailable["supported_assertion_score"] = "ASSERTION_REACHABILITY_UNVERIFIED"
            unavailable["effective_requirement_check_coverage"] = "ASSERTION_REACHABILITY_UNVERIFIED"
        evidence = {
            "collection_status": effective_collection_status,
            "collected_items": int(e["collected_item_count"] or 0),
            "collection_failure_reason": (None if effective_collection_status == "PASS"
                                           else effective_collection_status),
            "baseline_status": effective_baseline_status,
            "baseline_failure_reason": (None if effective_baseline_status == "COMPLETE"
                                         else effective_baseline_status),
            "item_outcomes": list(b["item_outcomes"].values()),
            "request_attempts": baseline_requests,
            # Semantic judgments are retained for all claims. Unreached source
            # assertions get no credit under the Stage 06 metrics contract.
            "reviewed_claims": [{"reviewed": True, "supported": (r["review_label"] == "supported" and r["reachability"] == "REACHED")} for r in reviewed],
            "effective_checks": effective, "unavailable_endpoints": unavailable,
            "mutation_results": [{"fault_id": f, "consumed": row["consumed"],
                                  "result": "KILLED" if qualified_kill(row, listed_faults[f], b, requests_by_id)
                                  else row["status"]} for f, row in sorted(results.items())],
        }
        scores = metric.score_operation_condition(evidence, answer)
        if b["baseline_pass"] is not None:
            require(scores[1]["score"] == float(b["baseline_pass"]), "Adapter disagrees with frozen baseline result")
        selected = selection[op]
        long.extend({"op_id": op, "pair_id": task["pair_id"], "task_id": task_id,
                     "family_id": selected["family_id"], "stratum": selected["stratum"],
                     "pi": float(selected["inclusion_probability"]), "condition": task["condition"], **s} for s in scores)
    indexed = keyed(long, lambda r: (r["op_id"], r["condition"], r["endpoint"]), "long scores")
    paired = []
    for op, selected in sorted(selection.items()):
        l0 = [indexed[op, "L0", ep] for ep in metric.PRIMARY_ENDPOINTS]
        l1 = [indexed[op, "L1", ep] for ep in metric.PRIMARY_ENDPOINTS]
        differences = metric.paired_metric_differences(l0, l1)
        for a, b in zip(l0, l1):
            ep = a["endpoint"]
            delta = differences[ep]
            paired.append({"op_id": op, "pair_id": selected["pair_id"], "family_id": selected["family_id"],
                           "stratum": selected["stratum"], "pi": float(selected["inclusion_probability"]),
                           "endpoint": ep, "L0": a["score"], "L1": b["score"], "delta": delta,
                           "common_evaluable": delta is not None, "L0_unavailable_reason": a["failure_reason"] if not a["observed"] else "",
                           "L1_unavailable_reason": b["failure_reason"] if not b["observed"] else "",
                           "estimand_type": a["estimand_type"]})
    return {"long": long, "paired": paired, "links": links, "execution": execution, "tasks": task_rows,
            "artifacts": artifact_rows, "requests": requests, "faults": faults, "variations": variations,
            "claims": claims, "baseline": baseline, "endpoints": metric.PRIMARY_ENDPOINTS}


sources = {s: published_pilot(s) for s in ("03", "04", "05", "06", "07", "08")}
verify_links(sources)
require(sources["08"].external.get("may_begin_notebook_09") is True
        and sources["08"].external.get("review_protocol_version") == PILOT_VERSION,
        "Attach the corrected Stage 08 release")
approved_packages = sources["06"].json("config/stage06_environment.json")["packages"]
for package in ("numpy", "scipy"):
    actual = runtime()[package]
    require(actual == approved_packages[package],
            f"Stage 06 registered {package}=={approved_packages[package]}, found {actual}. "
            "Install the registered versions and restart the kernel before scoring.")
registered = sources["06"].yaml("config/analysis_registered.yaml")
stats_source = sources["06"].read("src/statistics.py")
require(sha(stats_source) == registered["implementation_sha256"], "Stage 06 statistics implementation hash differs")
statistics = load_module("registered_statistics", stats_source)
selection = keyed(sources["05"].jsonl("partitions/pilot/final_selected_operations.jsonl"),
                  lambda r: r["op_id"], "final pilot selection")
strata = keyed(sources["05"].jsonl("partitions/pilot/final_stratum_sizes.jsonl"),
               lambda r: r["stratum"], "final pilot population")
shares = {h: v["W_h"] for h, v in strata.items()}
require(len(selection) == 100 and set(strata) == set(STRATA)
        and sources["05"].internal.get("human_review_complete") is True, "Pilot selection or oracle gate differs")
for item in selection.values():
    h = strata[item["stratum"]]
    require_math(item["inclusion_probability"], h["n_h"] / h["N_h"], "Pilot inclusion probability differs")
    require(item["final_status"] == "ELIGIBLE" and item["development_exposed"] is False, "Ineligible pilot member")
scoring_source = sources["06"].read("src/metrics.py")
unavailable_review, unavailable_review_bytes = load_unavailable_review()
result = score_partition(sources, selection, "pilot", scoring_source, unavailable_review)
require(len(result["long"]) == 1400 and len(result["paired"]) == 700, "Wrong pilot score grid")
if sources["08"].external_hash == PILOT_HASHES["08"]:
    require(len(result["claims"]) == 756 and Counter(
        row["baseline_status"] for row in result["execution"].values()) ==
        Counter(PASS=17, TEST_FAILURE=72, NOT_ELIGIBLE=95, EVALUATOR_UNAVAILABLE=16),
        "Corrected Stage 08 known audit counts differ")

SCORING_ADAPTER_SOURCE = '# Notebook 09 · Cell 02 · Reconcile and score the complete pilot ledger\n\nEVIDENCE = "data/evidence/"\n\n\ndef load_unavailable_review():\n    matches = sorted(INPUT.rglob("evaluator_unavailable_review_completed.csv"))\n    require(len(matches) == 1,\n            "Attach exactly one evaluator_unavailable_review_completed.csv")\n    raw = matches[0].read_bytes()\n    rows = csv_rows(raw)\n    required = {\n        "task_id", "op_id", "artifact_id", "collection_status", "baseline_status",\n        "request_ids", "reviewed_scenario_id", "baseline_id",\n        "spec_evidence_pointer", "oracle_evidence_pointer", "decision",\n        "rationale", "recommended_action",\n    }\n    require(rows and required <= set(rows[0]), "Unavailable review schema differs")\n    decisions = Counter(row["decision"] for row in rows)\n    require(len(rows) == 16 and decisions == Counter(\n        TEST_CONSTRUCTION_DEFECT=15, EVALUATOR_LIMITATION=1),\n        "Expected the completed 15 generated-test / 1 evaluator-limitation adjudication")\n    require(all(row["task_id"] and row["op_id"] and row["artifact_id"]\n                and row["request_ids"] and row["spec_evidence_pointer"]\n                and row["oracle_evidence_pointer"] and row["rationale"]\n                and row["recommended_action"]\n                and row["reviewed_scenario_id"] and row["baseline_id"]\n                for row in rows), "Unavailable review contains blank required evidence")\n    return keyed(rows, lambda row: row["task_id"], "unavailable adjudication"), raw\n\n\ndef reconcile_review(evaluation, claims):\n    require(evaluation.internal.get("checkpoint_c_complete") is True\n            and evaluation.external.get("checkpoint_c_complete") is True,\n            "Stage 08 has not completed Checkpoint C")\n    by_blind = keyed(claims, lambda r: r["blind_claim_id"], "reviewed claims")\n    require(all(k and r["review_status"] == "RECONCILED" for k, r in by_blind.items()), "Incomplete assertion review")\n    r1, r2, r3 = [keyed(evaluation.csv(f"reviews/completed/reviewer_{i}.csv"),\n                        lambda r: r["blind_claim_id"], f"Reviewer {i}") for i in (1, 2, 3)]\n    sample = evaluation.json(EVIDENCE + "reviewer_2_frozen_sample.json")\n    expected = sorted(random.Random(sample["seed"]).sample(sorted(by_blind), min(len(claims), 300)))\n    require(sample["sampled_ids"] == expected and r2.keys() == set(expected)\n            and r1.keys() == by_blind.keys(), "Incorrect frozen review coverage")\n    disagreements = {k for k in r2 if r1[k]["label"] != r2[k]["label"]}\n    require(r3.keys() == disagreements and all(r["adjudication_status"] == "COMPLETED" for r in r3.values()),\n            "Incomplete Reviewer 3 decisions")\n    labels = {"supported", "contradicted", "undocumented_or_unsupported", "ambiguous_or_unverifiable"}\n    for number, decisions in enumerate((r1, r2, r3), 1):\n        require(all(row["label"] in labels and row["reviewer_id"] == f"reviewer_{number}"\n                    and row["reasoning"].strip() for row in decisions.values()), "Invalid recorded review")\n    for k, row in by_blind.items():\n        chosen = r3.get(k, r1[k])\n        require(row["review_label"] == chosen["label"] and row["requirement_id"] == chosen["requirement_id"]\n                and row["review_evidence_pointer"] == chosen["evidence_pointer"],\n                "Final claim does not match the reconciled reviewer decision")\n    internal = keyed(evaluation.jsonl(EVIDENCE + "assertion_claims_internal.jsonl"),\n                     lambda r: r["claim_id"], "frozen assertion extraction")\n    require(internal.keys() == {r["claim_id"] for r in claims}\n            and all(all(row[k] == v for k, v in internal[row["claim_id"]].items()) for row in claims),\n            "Frozen assertion frame differs from reviewed frame")\n\n\ndef expected_oracles(release, selection, partition):\n    result = {}\n    record_types = {"scenarios": "scenario_id", "baselines": "baseline_id",\n                    "requirements": "requirement_id", "response_faults": "fault_id",\n                    "legitimate_variations": "variation_id"}\n    for op in selection:\n        root = f"oracles/frozen/{partition}/{op}"\n        records = {kind: release.jsonl(f"{root}/{kind}.jsonl") for kind in record_types}\n        for kind, key in record_types.items():\n            keyed(records[kind], lambda r: r[key], kind)\n        records["faults"] = records["response_faults"]\n        result[op] = records\n    return result\n\n\ndef qualified_kill(row, fault, baseline, requests):\n    if row["status"] != "KILLED_ASSERTION":\n        return False\n    target = row["replay_target"]\n    attempt = requests.get(row.get("target_baseline_request_id"))\n    item = target["pytest_item_id"]\n    require(baseline["baseline_pass"] is True and row["killed"] is True and row["consumed"] is True\n            and row["failure_attribution"] == "assertion" and attempt is not None\n            and attempt["status"] == "DELIVERED" and attempt["task_id"] == row["task_id"]\n            and attempt["pytest_item_id"] == item and item in row["attributed_assertion_item_ids"]\n            and baseline["item_outcomes"].get(item) == "passed"\n            and attempt.get("baseline_id") == fault["baseline_id"]\n            and any(x.get("replay_mutation_applied") and x.get("fault_id") == fault["fault_id"]\n                    and x.get("pytest_item_id") == item for x in row["request_attempts"]),\n            "A claimed kill lacks a qualified assertion-linked replay")\n    return True\n\n\ndef effective_check(claim, oracle, attempts, fault_rows):\n    if claim["review_label"] != "supported" or claim["reachability"] != "REACHED":\n        return None\n    req = claim["requirement_id"]\n    requirement = next((r for r in oracle["requirements"] if r["requirement_id"] == req), None)\n    require(requirement is not None and requirement["source_pointer"] == claim["review_evidence_pointer"],\n            "Supported claim lacks a corresponding frozen requirement")\n    reached = set(claim["reached_item_ids"])\n    delivered = [r for r in attempts if r["status"] == "DELIVERED"\n                 and r["pytest_item_id"] in reached\n                 and r.get("scenario_id") in claim["observed_scenario_ids"]]\n    if not delivered:\n        return None\n    linked_faults = {r["fault_id"] for r in oracle["faults"] if r["requirement_id"] == req\n                     and r.get("applicability") == "APPLICABLE"}\n    for fault_id in sorted(linked_faults):\n        row = fault_rows[fault_id]\n        if row["status"] != "KILLED_ASSERTION":\n            continue\n        target = row["replay_target"]\n        if not any(r["request_id"] == row["target_baseline_request_id"] for r in delivered):\n            continue\n        # Stage 08 names the failed item. Locate the actual failing source line\n        # before crediting this particular assertion.\n        location = re.compile(r"^work/generated_test\\.py:" + str(claim["line"])\n                              + r": in " + re.escape(target["pytest_item_id"].split("::")[-1].split("[")[0]) + r"\\s*$",\n                              re.MULTILINE)\n        if location.search(row["stdout"]) and any(\n                line.strip() == claim["source_span"].strip()\n                for line in row["stdout"].splitlines() if line.strip().startswith("assert ")):\n            return {"requirement_id": req, "reached": True, "grounded": True,\n                    "linkage_qualified": True, "evidence_type": "assertion_linked_frozen_fault"}\n    if linked_faults:\n        return None\n    # The registered semantic review can substantiate non-fault-linked checks\n    # if the assertion has a single unambiguous observed response context.\n    if len({(r["scenario_id"], r.get("response_status")) for r in delivered}) == 1:\n        return {"requirement_id": req, "reached": True, "grounded": True,\n                "linkage_qualified": True, "evidence_type": "reconciled_semantic_review"}\n    return None\n\n\ndef score_partition(sources, selection, partition, scoring_bytes, unavailable_review):\n    oracle, tasks, generation, evaluation = (sources[s] for s in ("05", "06", "07", "08"))\n    for release in (generation, evaluation):\n        require(release.read("config/metrics.yaml") == tasks.read("config/metrics.yaml")\n                and release.read("config/analysis_registered.yaml") == tasks.read("config/analysis_registered.yaml"),\n                "Frozen scoring or analysis contract differs")\n    require(scoring_bytes == tasks.read("src/metrics.py") == evaluation.read("src/metrics.py"),\n            "Scoring implementation differs from frozen Stage 06")\n    metric = load_module("frozen_metrics", scoring_bytes)\n    task_rows = keyed(tasks.csv("data/tasks.csv"), lambda r: r["task_id"], "tasks")\n    artifact_rows = keyed(generation.csv("data/artifacts.csv"), lambda r: r["task_id"], "raw-generation ledger")\n    execution = keyed(evaluation.csv(EVIDENCE + "execution.csv"), lambda r: r["task_id"], "execution")\n    baseline = keyed(evaluation.jsonl(EVIDENCE + "baseline_results.jsonl"), lambda r: r["task_id"], "baseline")\n    collection = keyed(evaluation.jsonl(EVIDENCE + "collection_results.jsonl"), lambda r: r["task_id"], "collection")\n    require(task_rows.keys() == artifact_rows.keys() == execution.keys() == baseline.keys() == collection.keys(),\n            "Expected task identities differ across Stages 06–08")\n    unavailable_tasks = {task_id for task_id in task_rows\n                         if execution[task_id]["collection_status"] == "EVALUATOR_UNAVAILABLE"\n                         or execution[task_id]["baseline_status"] == "EVALUATOR_UNAVAILABLE"}\n    require(set(unavailable_review) == unavailable_tasks,\n            "Completed adjudication must cover every and only evaluator-unavailable task")\n    for task_id, row in unavailable_review.items():\n        require(row["op_id"] == task_rows[task_id]["op_id"]\n                and row["artifact_id"] == artifact_rows[task_id]["artifact_id"]\n                and row["collection_status"] == execution[task_id]["collection_status"]\n                and row["baseline_status"] == execution[task_id]["baseline_status"],\n                "Adjudication identity or frozen status differs")\n    require(len(task_rows) == 2 * len(selection), "Expected one generation per condition and operation")\n    for op, operation_tasks in group(task_rows.values(), lambda r: r["op_id"]).items():\n        require(op in selection and len(operation_tasks) == 2\n                and {r["condition"] for r in operation_tasks} == {"L0", "L1"}\n                and {r["pair_id"] for r in operation_tasks} == {selection[op]["pair_id"]}\n                and len({r["seed"] for r in operation_tasks}) == 1, "Invalid task pair or paired seed")\n    require({r["op_id"] for r in task_rows.values()} == selection.keys(), "Task and selected-operation frames differ")\n    require(all(r["partition"] == partition and str(r["replicate"]) == "1" for r in task_rows.values()),\n            "Task partition or replicate differs")\n    inventory = keyed(evaluation.jsonl("data/inventory/input_artifact_inventory.jsonl"),\n                      lambda r: r["task_id"], "execution input inventory")\n    require(inventory.keys() == task_rows.keys(), "Execution inventory differs")\n    for k, row in artifact_rows.items():\n        require(inventory[k]["raw_sha256"] == (row["raw_hash"] or None)\n                and inventory[k]["extracted_sha256"] == (row["extracted_hash"] or None)\n                and inventory[k]["extraction_status"] == row["extraction_status"],\n                "Evaluation consumed different generation evidence")\n    claims = evaluation.jsonl(EVIDENCE + "assertion_claims.jsonl")\n    reconcile_review(evaluation, claims)\n    by_task_claim = group(claims, lambda r: r["task_id"])\n    requests = evaluation.jsonl(EVIDENCE + "request_attempts.jsonl")\n    requests_by_id = keyed(requests, lambda r: r["request_id"], "request attempts")\n    by_task_requests = group(requests, lambda r: r["task_id"])\n    faults = evaluation.jsonl(EVIDENCE + "mutation_results.jsonl")\n    variations = evaluation.jsonl(EVIDENCE + "variation_results.jsonl")\n    frozen = keyed(evaluation.jsonl(EVIDENCE + "opportunities_frozen.jsonl"),\n                   lambda r: r["opportunity_id"], "frozen opportunities")\n    outcomes = keyed(evaluation.jsonl(EVIDENCE + "opportunity_outcomes.jsonl"),\n                     lambda r: r["opportunity_id"], "opportunity results")\n    require(frozen.keys() == outcomes.keys() and all(all(outcomes[k][f] == v for f, v in row.items())\n            for k, row in frozen.items()), "Frozen replay ledger differs from its reconciliation")\n    for kind, rows, id_key in (("fault", faults, "fault_id"), ("variation", variations, "variation_id")):\n        lookup = keyed(rows, lambda r: r["opportunity_id"], f"{kind} results")\n        require(lookup.keys() == {k for k, row in frozen.items() if row["kind"] == kind}\n                and all(row["task_id"] == frozen[k]["task_id"]\n                        and row[id_key] == frozen[k]["oracle_record_id"]\n                        and row["status"] == outcomes[k]["final_status"] for k, row in lookup.items()),\n                f"{kind} reconciliation mismatch")\n    by_task_fault = group(faults, lambda r: r["task_id"])\n    by_task_opportunity = group(frozen.values(), lambda r: r["task_id"])\n    require(set(by_task_claim) | set(by_task_requests) <= task_rows.keys()\n            and by_task_opportunity.keys() == task_rows.keys(), "Evidence contains unknown or missing tasks")\n    oracles = expected_oracles(oracle, selection, partition)\n    long, links = [], []\n    for task_id, task in sorted(task_rows.items()):\n        op = task["op_id"]\n        answer = oracles[op]\n        expected = {("baseline", r["baseline_id"]) for r in answer["baselines"]}\n        expected |= {("scenario", r["scenario_id"]) for r in answer["scenarios"] if r.get("applicability", "APPLICABLE") == "APPLICABLE"}\n        expected |= {("fault", r["fault_id"]) for r in answer["faults"] if r.get("applicability", "APPLICABLE") == "APPLICABLE"}\n        expected |= {("variation", r["variation_id"]) for r in answer["legitimate_variations"] if r.get("applicability", "APPLICABLE") == "APPLICABLE"}\n        actual = [(r["kind"], r["oracle_record_id"]) for r in by_task_opportunity[task_id]]\n        require(len(actual) == len(expected) and set(actual) == expected\n                and all(r["op_id"] == op for r in by_task_opportunity[task_id]), "Incomplete oracle opportunity frame")\n        b, e = baseline[task_id], execution[task_id]\n        require(e["baseline_status"] == b["status"] and e["collection_status"] == collection[task_id]["status"],\n                "Execution summaries disagree with detailed results")\n        require(b["status"] != "EVALUATOR_FAILURE" and e["collection_status"] != "EVALUATOR_FAILURE",\n                "Evaluator failure requires a documented rerun")\n        require(int(e["collected_item_count"] or 0) == len(collection[task_id]["item_ids"]),\n                "Collected item count differs")\n        adjudication = unavailable_review.get(task_id)\n        generated_output_failure = bool(\n            adjudication and adjudication["decision"] == "TEST_CONSTRUCTION_DEFECT")\n        effective_collection_status = (\n            "GENERATED_OUTPUT_FAILURE"\n            if generated_output_failure and e["collection_status"] == "EVALUATOR_UNAVAILABLE"\n            else e["collection_status"]\n        )\n        effective_baseline_status = (\n            "GENERATED_OUTPUT_FAILURE"\n            if generated_output_failure and b["status"] == "EVALUATOR_UNAVAILABLE"\n            else ("COMPLETE" if b["status"] == "PASS" else b["status"])\n        )\n        seen = by_task_requests[task_id]\n        require(all(r["op_id"] == op for r in seen), "Request evidence points to another operation")\n        applicable_scenarios = {r["scenario_id"] for r in answer["scenarios"] if r.get("applicability", "APPLICABLE") == "APPLICABLE"}\n        baseline_requests = []\n        for r in seen:\n            cl = r.get("classification") or {}\n            scenario = r.get("scenario_id")\n            classifiable = cl.get("classifiable") is True or r.get("reason_code") == "REQUEST_OUTSIDE_TARGET_OPERATION"\n            baseline_requests.append({"attempt_id": r["request_id"], "classifiable": classifiable,\n                                      "matched_intent": classifiable and scenario in applicable_scenarios,\n                                      "scenario_id": scenario if classifiable else None})\n        reviewed = by_task_claim[task_id]\n        require(all(r["op_id"] == op and r["condition"] == task["condition"]\n                    and r["artifact_id"] == artifact_rows[task_id]["artifact_id"] for r in reviewed),\n                "Reviewed assertion points to another generation")\n        listed_faults = {r["fault_id"]: r for r in answer["faults"] if r.get("applicability", "APPLICABLE") == "APPLICABLE"}\n        results = keyed(by_task_fault[task_id], lambda r: r["fault_id"], "task fault results")\n        effective = [link for claim in reviewed if (link := effective_check(claim, answer, seen, results))]\n        links.extend({"task_id": task_id, **link} for link in effective)\n        unavailable = {}\n        if (not generated_output_failure\n                and any(r["status"] == "EVALUATOR_UNAVAILABLE" for r in results.values())):\n            unavailable["unconditional_response_fault_detection"] = "FAULT_OR_BASELINE_EVALUATOR_UNAVAILABLE"\n        if (not generated_output_failure\n                and any(r["reachability"] == "UNVERIFIED" for r in reviewed)\n                and e["collection_status"] == "PASS"):\n            unavailable["supported_assertion_score"] = "ASSERTION_REACHABILITY_UNVERIFIED"\n            unavailable["effective_requirement_check_coverage"] = "ASSERTION_REACHABILITY_UNVERIFIED"\n        evidence = {\n            "collection_status": effective_collection_status,\n            "collected_items": int(e["collected_item_count"] or 0),\n            "collection_failure_reason": (None if effective_collection_status == "PASS"\n                                           else effective_collection_status),\n            "baseline_status": effective_baseline_status,\n            "baseline_failure_reason": (None if effective_baseline_status == "COMPLETE"\n                                         else effective_baseline_status),\n            "item_outcomes": list(b["item_outcomes"].values()),\n            "request_attempts": baseline_requests,\n            # Semantic judgments are retained for all claims. Unreached source\n            # assertions get no credit under the Stage 06 metrics contract.\n            "reviewed_claims": [{"reviewed": True, "supported": (r["review_label"] == "supported" and r["reachability"] == "REACHED")} for r in reviewed],\n            "effective_checks": effective, "unavailable_endpoints": unavailable,\n            "mutation_results": [{"fault_id": f, "consumed": row["consumed"],\n                                  "result": "KILLED" if qualified_kill(row, listed_faults[f], b, requests_by_id)\n                                  else row["status"]} for f, row in sorted(results.items())],\n        }\n        scores = metric.score_operation_condition(evidence, answer)\n        if b["baseline_pass"] is not None:\n            require(scores[1]["score"] == float(b["baseline_pass"]), "Adapter disagrees with frozen baseline result")\n        selected = selection[op]\n        long.extend({"op_id": op, "pair_id": task["pair_id"], "task_id": task_id,\n                     "family_id": selected["family_id"], "stratum": selected["stratum"],\n                     "pi": float(selected["inclusion_probability"]), "condition": task["condition"], **s} for s in scores)\n    indexed = keyed(long, lambda r: (r["op_id"], r["condition"], r["endpoint"]), "long scores")\n    paired = []\n    for op, selected in sorted(selection.items()):\n        l0 = [indexed[op, "L0", ep] for ep in metric.PRIMARY_ENDPOINTS]\n        l1 = [indexed[op, "L1", ep] for ep in metric.PRIMARY_ENDPOINTS]\n        differences = metric.paired_metric_differences(l0, l1)\n        for a, b in zip(l0, l1):\n            ep = a["endpoint"]\n            delta = differences[ep]\n            paired.append({"op_id": op, "pair_id": selected["pair_id"], "family_id": selected["family_id"],\n                           "stratum": selected["stratum"], "pi": float(selected["inclusion_probability"]),\n                           "endpoint": ep, "L0": a["score"], "L1": b["score"], "delta": delta,\n                           "common_evaluable": delta is not None, "L0_unavailable_reason": a["failure_reason"] if not a["observed"] else "",\n                           "L1_unavailable_reason": b["failure_reason"] if not b["observed"] else "",\n                           "estimand_type": a["estimand_type"]})\n    return {"long": long, "paired": paired, "links": links, "execution": execution, "tasks": task_rows,\n            "artifacts": artifact_rows, "requests": requests, "faults": faults, "variations": variations,\n            "claims": claims, "baseline": baseline, "endpoints": metric.PRIMARY_ENDPOINTS}\n'


In [3]:
# Notebook 09 · Cell 03 · Summarize paired evidence and disagreement

summary, stratum_summary, disagreements = [], [], []
by_endpoint = group(result["paired"], lambda r: r["endpoint"])
for endpoint, rows in by_endpoint.items():
    observed = [r for r in rows if r["common_evaluable"]]
    conditional = endpoint == "request_intent_match"
    complete = len(observed) == len(rows)
    families = Counter(r["family_id"] for r in observed)
    weights = {r["op_id"]: 1 / r["pi"] for r in observed}
    weight_total = sum(weights.values())
    stratum_rows = group(observed, lambda r: r["stratum"])
    target_weights = ({h: sum(weights[r["op_id"]] for r in subset) / weight_total
                       for h, subset in stratum_rows.items()} if conditional and observed else shares)
    deltas = [r["delta"] for r in observed]
    def weighted_condition(key):
        if not observed or (not complete and not conditional):
            return None
        return sum(target_weights[h] * sum(weights[r["op_id"]] * r[key] for r in sub)
                   / sum(weights[r["op_id"]] for r in sub) for h, sub in stratum_rows.items())
    a, b = (weighted_condition(k) for k in ("L0", "L1"))
    status = ("AVAILABLE_CONDITIONAL" if conditional and observed else "AVAILABLE_COMPLETE"
              if complete else "INCOMPLETE_EVIDENCE")
    record = {"endpoint": endpoint, "total_pairs": len(rows), "common_evaluable_pairs": len(observed),
              "estimand_type": rows[0]["estimand_type"], "status": status, "families": len(families),
              "largest_family": max(families.values(), default=0),
              "family_concentration_m2_over_m": sum(n*n for n in families.values()) / len(observed) if observed else None,
              "L0_common_pair_mean": mean(r["L0"] for r in observed) if observed else None,
              "L1_common_pair_mean": mean(r["L1"] for r in observed) if observed else None,
              "mean_delta_common_pairs": mean(deltas) if deltas else None,
              "sd_delta_common_pairs": stdev(deltas) if len(deltas) > 1 else None,
              "weighted_L0": a, "weighted_L1": b, "weighted_delta": b-a if a is not None and b is not None else None,
              "discordance": None, "discordance_ci_low": None, "discordance_ci_high": None}
    if endpoint in {"collection_success", "baseline_pass"}:
        counts = Counter((int(r["L0"]), int(r["L1"])) for r in observed)
        q = (counts[0, 1] + counts[1, 0]) / len(observed) if observed else None
        interval = {}
        if len(families) > 1 and any(0 < sum(r["delta"] != 0 for r in subset) < len(subset)
                                                  for subset in stratum_rows.values()):
            interval = statistics.family_multiplier_interval(
                [{"stratum": r["stratum"], "family_id": r["family_id"], "pi": r["pi"],
                  "d": float(r["delta"] != 0)} for r in observed],
                shares, seed=registered["analysis_seed"], draws=10000, conditional=not complete)
        record.update(discordance=q, discordance_ci_low=interval.get("lower"),
                      discordance_ci_high=interval.get("upper"))
        q_weighted = (statistics.weighted_stratum_effect(
            [{**r, "d": float(r["delta"] != 0)} for r in observed], shares)["effect"] if complete else None)
        disagreements.append({"endpoint": endpoint, "both_fail": counts[0,0], "L0_only": counts[1,0],
                              "L1_only": counts[0,1], "both_pass": counts[1,1], "unavailable_pairs": len(rows)-len(observed),
                              "discordance_common_pairs": q, "weighted_discordance": q_weighted,
                              "family_multiplier_interval_low": interval.get("lower"),
                              "family_multiplier_interval_high": interval.get("upper"),
                              "interval_target": ("full_population" if complete else "common_evaluable_pairs_descriptive") if interval else "",
                              "precision_target": ("MET" if interval and complete and
                                  max(q_weighted-interval["lower"], interval["upper"]-q_weighted) <= 0.10
                                  else "NOT_MET" if interval and complete else "NOT_CERTIFIED")})
    summary.append(record)
    for h in STRATA:
        total = [r for r in rows if r["stratum"] == h]
        subset = stratum_rows[h]
        stratum_summary.append({"endpoint": endpoint, "stratum": h, "total_pairs": len(total),
                                "common_evaluable_pairs": len(subset),
                                "total_families": len({r["family_id"] for r in total}),
                                "common_evaluable_families": len({r["family_id"] for r in subset}),
                                "L0_mean_common_pairs": mean(r["L0"] for r in subset) if subset else None,
                                "L1_mean_common_pairs": mean(r["L1"] for r in subset) if subset else None,
                                "mean_delta_common_pairs": mean(r["delta"] for r in subset) if subset else None,
                                "sd_delta_common_pairs": stdev(r["delta"] for r in subset) if len(subset)>1 else None})
blockers = [r["endpoint"] for r in summary if r["status"] == "INCOMPLETE_EVIDENCE"]
print("Pilot pairs by endpoint", {r["endpoint"]: r["common_evaluable_pairs"] for r in summary})
print("Endpoints with incomplete full-population evidence", blockers)
for row in summary:
    print(row["endpoint"], "paired descriptive mean", row["mean_delta_common_pairs"])


Pilot pairs by endpoint {'collection_success': 100, 'baseline_pass': 99, 'request_intent_match': 21, 'selected_scenario_coverage': 99, 'supported_assertion_score': 99, 'effective_requirement_check_coverage': 99, 'unconditional_response_fault_detection': 99}
Endpoints with incomplete full-population evidence ['baseline_pass', 'selected_scenario_coverage', 'supported_assertion_score', 'effective_requirement_check_coverage', 'unconditional_response_fault_detection']
collection_success paired descriptive mean 0.06
baseline_pass paired descriptive mean -0.0707070707070707
request_intent_match paired descriptive mean -0.01904761904761905
selected_scenario_coverage paired descriptive mean -0.007864194845853538
supported_assertion_score paired descriptive mean -0.00202020202020202
effective_requirement_check_coverage paired descriptive mean -0.00015076134479119553
unconditional_response_fault_detection paired descriptive mean -0.05353461515742217


In [4]:
# Notebook 09 · Cell 04 · Export the verified Stage 09 package

review = sources["08"].read(EVIDENCE + "review_reliability.csv")
report = {
    "pipeline_sections": ["11.2", "14", "15", "20", "22.4"],
    "stage00_internal_manifest_sha256": sources["06"].internal["upstream_manifest_sha256"]["00"],
    "stage01_02_declared_by_stage03": {k:v for k,v in sources["03"].internal["upstream_manifest_sha256"].items() if k in {"01","02"}},
    "sources": {s: {"external_sha256": r.external_hash, "internal_sha256": r.internal_hash,
                    "registered_members_read_and_verified": len(r.checked)} for s,r in sources.items()},
    "operation_count": len(selection), "claim_count": len(result["claims"]),
    "reachability_counts": dict(Counter(r["reachability"] for r in result["claims"])),
    "full_population_endpoints_with_incomplete_evidence": blockers,
    "unavailable_adjudication": {"generated_output_failure": 15,
                                  "evaluator_limitation": 1,
                                  "review_sha256": sha(unavailable_review_bytes)},
    "main_sample_sizing_authorized": False,
    "runtime": runtime(),
}
files = {
    "data/evidence/scores_long.csv": csv_bytes(result["long"]),
    "data/evidence/paired_scores.csv": csv_bytes(result["paired"]),
    "data/evidence/evaluator_unavailable_review_completed.csv": unavailable_review_bytes,
    "reports/pilot_endpoint_summary.csv": csv_bytes(summary),
    "reports/pilot_stratum_summary.csv": csv_bytes(stratum_summary),
    "reports/pilot_binary_disagreements.csv": csv_bytes(disagreements),
    "reports/review_reliability.csv": review,
    "reports/effective_check_linkage.csv": csv_bytes(result["links"],
        ["task_id", "requirement_id", "reached", "grounded", "linkage_qualified", "evidence_type"]),
    "reports/pilot_score_audit.json": json_bytes(report),
    "config/analysis_registered.yaml": sources["06"].read("config/analysis_registered.yaml"),
    "config/metrics.yaml": sources["06"].read("config/metrics.yaml"),
    "src/metrics.py": scoring_source,
    "src/scoring_adapter.py": SCORING_ADAPTER_SOURCE.encode("utf-8"),
    "src/statistics.py": stats_source,
}
write_release("09", "pilot", files, sources,
              {"operation_count": 100, "task_count": 200, "paired_score_rows": 700,
               "main_sample_sizing_authorized": False,
               "analysis_mode": "DESCRIPTIVE_PILOT_ADJUDICATED",
               "generated_output_failure_adjudications": 15,
               "retained_evaluator_limitations": 1,
               "full_population_endpoints_with_incomplete_evidence": blockers})
print("Checkpoint: pilot scoring complete. Resolve the planning inputs in Notebook 10.")


Stage 09 pilot release verified at /kaggle/working/stage_09_pilot_dataset
Checkpoint: pilot scoring complete. Resolve the planning inputs in Notebook 10.
